# Module 083: Introduction to SQL from Python

Phase 9: Databases & Web Apps | Duration: 2 hours

## Setup: Sample Database

In [ ]:
import sqlite3

conn: sqlite3.Connection = sqlite3.connect(':memory:')
cursor: sqlite3.Cursor = conn.cursor()

cursor.executescript('''
    CREATE TABLE customers (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        city TEXT
    );
    CREATE TABLE orders (
        id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        product TEXT,
        amount REAL,
        FOREIGN KEY (customer_id) REFERENCES customers(id)
    );
    INSERT INTO customers VALUES (1, 'Alice', 'NYC'), (2, 'Bob', 'LA'), (3, 'Carol', 'NYC');
    INSERT INTO orders VALUES
        (1, 1, 'Laptop', 1200),
        (2, 1, 'Mouse', 25),
        (3, 2, 'Keyboard', 80),
        (4, 3, 'Monitor', 300),
        (5, 3, 'USB Hub', 40);
''')
print('Sample database created')

## SELECT with WHERE and ORDER BY

In [ ]:
cursor.execute('SELECT * FROM orders WHERE amount > ? ORDER BY amount DESC', (50,))
for row in cursor.fetchall():
    print(row)

## GROUP BY and Aggregate Functions

In [ ]:
cursor.execute('''
    SELECT customer_id, COUNT(*) as order_count, SUM(amount) as total_spent
    FROM orders
    GROUP BY customer_id
''')
for row in cursor.fetchall():
    print(f'Customer {row[0]}: {row[1]} orders, ${row[2]:.2f} total')

## HAVING Clause

In [ ]:
cursor.execute('''
    SELECT customer_id, SUM(amount) as total
    FROM orders
    GROUP BY customer_id
    HAVING total > 100
''')
print('Customers with total > 100:', cursor.fetchall())

## JOINs: INNER and LEFT

In [ ]:
# INNER JOIN - only customers with orders
cursor.execute('''
    SELECT c.name, o.product, o.amount
    FROM customers c
    INNER JOIN orders o ON c.id = o.customer_id
''')
print('INNER JOIN:')
for row in cursor.fetchall():
    print(f'  {row[0]}: {row[1]} (${row[2]:.2f})')

In [ ]:
# LEFT JOIN - all customers, even without orders
# (Add a customer with no orders first)
cursor.execute("INSERT INTO customers VALUES (4, 'David', 'Chicago')")
conn.commit()

cursor.execute('''
    SELECT c.name, o.product, o.amount
    FROM customers c
    LEFT JOIN orders o ON c.id = o.customer_id
''')
print('LEFT JOIN:')
for row in cursor.fetchall():
    product: str = row[1] if row[1] else 'No orders'
    amount: str = f'(${row[2]:.2f})' if row[2] else ''
    print(f'  {row[0]}: {product} {amount}')

## Row Factory: Dict-like Access

In [ ]:
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

cursor.execute('SELECT * FROM customers')
for row in cursor.fetchall():
    print(f'Name: {row["name"]}, City: {row["city"]}')

## fetchone and fetchmany

In [ ]:
cursor.execute('SELECT * FROM orders')

print('fetchone:')
row = cursor.fetchone()
while row:
    print(f'  Order {row["id"]}: {row["product"]}')
    row = cursor.fetchone()

cursor.execute('SELECT * FROM orders')
print('\nfetchmany(2):')
batch = cursor.fetchmany(2)
while batch:
    print(f'  Batch: {[dict(r)["product"] for r in batch]}')
    batch = cursor.fetchmany(2)

## Cleanup

In [ ]:
conn.close()